In [18]:
# Import getpass so the Neo4j password is entered securely rather than
# stored directly inside the notebook source.
import getpass

# Import pathlib for portable project-relative file paths.
from pathlib import Path

# Import pandas for loading and validating the exported CSV files.
import pandas as pd

# Import the official Neo4j Python driver.
from neo4j import GraphDatabase


# Resolve paths relative to the notebook's location.
DATA_DIR = Path("../data")

ENDPOINT_SECURITY_PATH = DATA_DIR / "endpoint_security.csv"
ARTIFACT_TABLE_PATH = DATA_DIR / "artifact_level_table.csv"
EDGE_TABLE_PATH = DATA_DIR / "endpoint_artifact_edges.csv"


# Neo4j connection configuration used by the project.
URI = "neo4j://127.0.0.1:7687"
USERNAME = "neo4j"
DATABASE = "indianarmygraph"


print("Endpoint security:", ENDPOINT_SECURITY_PATH.resolve())
print("Artifact table:", ARTIFACT_TABLE_PATH.resolve())
print("Edge table:", EDGE_TABLE_PATH.resolve())
print("Neo4j database:", DATABASE)

Endpoint security: C:\Users\Asus\Desktop\web_activity\data\endpoint_security.csv
Artifact table: C:\Users\Asus\Desktop\web_activity\data\artifact_level_table.csv
Edge table: C:\Users\Asus\Desktop\web_activity\data\endpoint_artifact_edges.csv
Neo4j database: indianarmygraph


In [19]:
# Prompt securely for the Neo4j password.
PASSWORD = getpass.getpass("Enter the Neo4j password: ")

# Create the Neo4j driver used throughout the notebook.
driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
)

# Verify connectivity before modifying the database.
driver.verify_connectivity()

print("Neo4j connection successful.")

Neo4j connection successful.


In [20]:
# Load the complete raw security telemetry.
security_df = pd.read_csv(ENDPOINT_SECURITY_PATH)

# Load the validated endpoint-artifact relationship table.
edge_df = pd.read_csv(EDGE_TABLE_PATH)

# Load the artifact-level table containing artifact-wide properties.
artifact_df = pd.read_csv(ARTIFACT_TABLE_PATH)


# Extract every endpoint present in the raw telemetry.
all_endpoint_ids = sorted(
    security_df["endpoint_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)


print("=" * 70)
print("SOURCE DATA VALIDATION")
print("=" * 70)

print("Raw security rows:", len(security_df))
print("Raw endpoints:", len(all_endpoint_ids))
print("Endpoint-artifact relationships:", len(edge_df))
print("Artifact rows:", len(artifact_df))

SOURCE DATA VALIDATION
Raw security rows: 23540
Raw endpoints: 247
Endpoint-artifact relationships: 72
Artifact rows: 60


In [21]:
# Verify that the relationship table contains the columns required
# by the final Neo4j relationship model.
required_edge_columns = {
    "endpoint_id",
    "sha256",
    "file_names",
    "record_count",
    "first_event",
    "last_event",
    "file_name_count",
    "malicious_detected",
    "malicious_detection_count",
    "malicious_file_names",
    "first_malicious_detection",
    "last_malicious_detection",
    "first_quarantine_time",
    "last_quarantine_time",
}

missing_edge_columns = (
    required_edge_columns - set(edge_df.columns)
)

if missing_edge_columns:
    raise ValueError(
        "The relationship CSV is missing required columns: "
        f"{sorted(missing_edge_columns)}"
    )


# Verify that every raw endpoint is represented in the endpoint population.
assert len(all_endpoint_ids) == 247, (
    f"Expected 247 raw endpoints, found {len(all_endpoint_ids)}."
)

# Verify the relationship count established by the previous notebook.
assert len(edge_df) == 72, (
    f"Expected 72 endpoint-artifact relationships, found {len(edge_df)}."
)

print("Source schema validation passed.")

Source schema validation passed.


In [22]:
# Remove the previous graph so stale malicious-only nodes and relationships
# cannot remain after the corrected import.
reset_query = """
MATCH (n)
DETACH DELETE n
"""

driver.execute_query(
    reset_query,
    database_=DATABASE,
)

print("Existing graph cleared.")

Existing graph cleared.


In [23]:
# Enforce uniqueness for endpoint identifiers.
# The constraint also provides indexed lookup support for Endpoint nodes.
driver.execute_query(
    """
    CREATE CONSTRAINT endpoint_id_unique IF NOT EXISTS
    FOR (e:Endpoint)
    REQUIRE e.endpoint_id IS UNIQUE
    """,
    database_=DATABASE,
)

# Enforce uniqueness for SHA-256 artifact identifiers.
# This guarantees one Artifact node per unique SHA-256.
driver.execute_query(
    """
    CREATE CONSTRAINT artifact_sha256_unique IF NOT EXISTS
    FOR (a:Artifact)
    REQUIRE a.sha256 IS UNIQUE
    """,
    database_=DATABASE,
)

print("Neo4j constraints verified.")

Neo4j constraints verified.


In [24]:
# Import every endpoint found in the raw telemetry.
# This deliberately includes endpoints that have no parseable artifact.
endpoint_records = [
    {"endpoint_id": endpoint_id}
    for endpoint_id in all_endpoint_ids
]


endpoint_query = """
UNWIND $endpoints AS row

MERGE (e:Endpoint {
    endpoint_id: row.endpoint_id
})
"""


driver.execute_query(
    endpoint_query,
    endpoints=endpoint_records,
    database_=DATABASE,
)

print(
    f"Imported {len(endpoint_records)} Endpoint nodes."
)

Imported 247 Endpoint nodes.


In [25]:
# Convert pipe-delimited CSV list fields back into Python lists
# before sending them to Neo4j.
def split_pipe_list(value):
    """Convert a pipe-delimited CSV field into a clean Python list."""

    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        item.strip()
        for item in value.split("|")
        if item.strip()
    ]


# Normalize the artifact table's list-valued columns.
artifact_df["file_names"] = (
    artifact_df["file_names"]
    .apply(split_pipe_list)
)

artifact_df["endpoint_ids"] = (
    artifact_df["endpoint_ids"]
    .apply(split_pipe_list)
)


print(
    "Artifacts prepared:",
    len(artifact_df),
)

Artifacts prepared: 60


In [26]:
# Convert relationship list-valued columns back into Python lists.
edge_df["file_names"] = (
    edge_df["file_names"]
    .apply(split_pipe_list)
)

edge_df["malicious_file_names"] = (
    edge_df["malicious_file_names"]
    .apply(split_pipe_list)
)


# Normalize the malicious flag because CSV loading may represent booleans
# as strings depending on how the file was written.
edge_df["malicious_detected"] = (
    edge_df["malicious_detected"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)


# Normalize integer relationship properties.
edge_df["record_count"] = (
    edge_df["record_count"]
    .fillna(0)
    .astype(int)
)

edge_df["file_name_count"] = (
    edge_df["file_name_count"]
    .fillna(0)
    .astype(int)
)

edge_df["malicious_detection_count"] = (
    edge_df["malicious_detection_count"]
    .fillna(0)
    .astype(int)
)


# Convert timestamp columns into Python datetime objects.
# Neo4j accepts Python datetime values directly.
datetime_columns = [
    "first_event",
    "last_event",
    "first_malicious_detection",
    "last_malicious_detection",
    "first_quarantine_time",
    "last_quarantine_time",
]

for column in datetime_columns:
    edge_df[column] = pd.to_datetime(
        edge_df[column],
        errors="coerce",
    )


print("Relationship data prepared.")

Relationship data prepared.


In [27]:
# Convert pandas artifact records into ordinary Python dictionaries.
artifact_records = artifact_df.to_dict("records")


artifact_query = """
UNWIND $artifacts AS row

MERGE (a:Artifact {
    sha256: row.sha256
})

SET
    a.file_names = row.file_names,
    a.file_name_count = row.file_name_count,
    a.record_count = row.record_count,
    a.first_event = row.first_event,
    a.last_event = row.last_event,
    a.endpoint_count = row.endpoint_count,
    a.endpoint_ids = row.endpoint_ids
"""


driver.execute_query(
    artifact_query,
    artifacts=artifact_records,
    database_=DATABASE,
)

print(
    f"Imported {len(artifact_records)} Artifact nodes."
)

Imported 60 Artifact nodes.


In [28]:
# Convert pandas datetime values into Python datetime values or None.
# This avoids passing pandas NaT values into the Neo4j driver.
def neo4j_datetime(value):
    """Convert a pandas timestamp into a Neo4j-compatible Python datetime."""

    if pd.isna(value):
        return None

    return value.to_pydatetime()


for column in datetime_columns:
    edge_df[column] = (
        edge_df[column]
        .apply(neo4j_datetime)
    )


# Replace pandas/numpy scalar values with native Python types where needed.
edge_df["endpoint_id"] = (
    edge_df["endpoint_id"]
    .astype(str)
)

edge_df["sha256"] = (
    edge_df["sha256"]
    .astype(str)
    .str.lower()
)

edge_df["record_count"] = (
    edge_df["record_count"]
    .astype(int)
)

edge_df["file_name_count"] = (
    edge_df["file_name_count"]
    .astype(int)
)

edge_df["malicious_detection_count"] = (
    edge_df["malicious_detection_count"]
    .astype(int)
)

edge_df["malicious_detected"] = (
    edge_df["malicious_detected"]
    .astype(bool)
)


edge_records = edge_df.to_dict("records")

print(
    "Prepared relationship records:",
    len(edge_records),
)

Prepared relationship records: 72


In [29]:
# Create one OBSERVED relationship for each unique endpoint-artifact pair.
#
# Malicious evidence is stored on the relationship because the security
# classification belongs to the observation context of that endpoint-artifact
# pair rather than globally to the Artifact node.
relationship_query = """
UNWIND $edges AS row

MATCH (e:Endpoint {
    endpoint_id: row.endpoint_id
})

MATCH (a:Artifact {
    sha256: row.sha256
})

MERGE (e)-[r:OBSERVED]->(a)

SET
    r.record_count = row.record_count,
    r.first_event = row.first_event,
    r.last_event = row.last_event,

    r.malicious_detected = row.malicious_detected,
    r.malicious_detection_count = row.malicious_detection_count,
    r.malicious_file_names = row.malicious_file_names,

    r.first_malicious_detection = row.first_malicious_detection,
    r.last_malicious_detection = row.last_malicious_detection,

    r.first_quarantine_time = row.first_quarantine_time,
    r.last_quarantine_time = row.last_quarantine_time
"""


driver.execute_query(
    relationship_query,
    edges=edge_records,
    database_=DATABASE,
)

print(
    f"Imported {len(edge_records)} OBSERVED relationships."
)

Imported 72 OBSERVED relationships.


In [30]:
# Count Endpoint nodes.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    RETURN count(e) AS endpoint_count
    """,
    database_=DATABASE,
)

endpoint_count = result.records[0]["endpoint_count"]


# Count Artifact nodes.
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    RETURN count(a) AS artifact_count
    """,
    database_=DATABASE,
)

artifact_count = result.records[0]["artifact_count"]


# Count OBSERVED relationships.
result = driver.execute_query(
    """
    MATCH ()-[r:OBSERVED]->()
    RETURN count(r) AS relationship_count
    """,
    database_=DATABASE,
)

relationship_count = result.records[0]["relationship_count"]


print("=" * 70)
print("GRAPH COUNTS")
print("=" * 70)

print("Endpoint nodes:", endpoint_count)
print("Artifact nodes:", artifact_count)
print("OBSERVED relationships:", relationship_count)

GRAPH COUNTS
Endpoint nodes: 247
Artifact nodes: 60
OBSERVED relationships: 72


In [31]:
# Confirm that every endpoint from the raw dataset exists in Neo4j.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    RETURN collect(e.endpoint_id) AS endpoints
    """,
    database_=DATABASE,
)

neo4j_endpoints = set(
    result.records[0]["endpoints"]
)

raw_endpoint_set = set(all_endpoint_ids)

missing_endpoints = sorted(
    raw_endpoint_set - neo4j_endpoints
)

unexpected_endpoints = sorted(
    neo4j_endpoints - raw_endpoint_set
)


print("=" * 70)
print("ENDPOINT COVERAGE")
print("=" * 70)

print("Raw endpoints:", len(raw_endpoint_set))
print("Neo4j endpoints:", len(neo4j_endpoints))
print("Missing endpoints:", len(missing_endpoints))
print("Unexpected endpoints:", len(unexpected_endpoints))

if missing_endpoints:
    print("\nMissing endpoints:")
    print(missing_endpoints)

if unexpected_endpoints:
    print("\nUnexpected endpoints:")
    print(unexpected_endpoints)

ENDPOINT COVERAGE
Raw endpoints: 247
Neo4j endpoints: 247
Missing endpoints: 0
Unexpected endpoints: 0


In [32]:
# Count endpoint-artifact relationships carrying explicit malicious evidence.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WHERE r.malicious_detected = true

    RETURN
        count(r) AS malicious_relationship_count,
        count(DISTINCT e) AS malicious_endpoint_count,
        count(DISTINCT a) AS malicious_artifact_count
    """,
    database_=DATABASE,
)

malicious_summary = dict(result.records[0])


print("=" * 70)
print("MALICIOUS EVIDENCE VALIDATION")
print("=" * 70)

print(
    "Malicious relationships:",
    malicious_summary["malicious_relationship_count"],
)

print(
    "Malicious endpoints:",
    malicious_summary["malicious_endpoint_count"],
)

print(
    "Malicious artifacts:",
    malicious_summary["malicious_artifact_count"],
)

MALICIOUS EVIDENCE VALIDATION
Malicious relationships: 60
Malicious endpoints: 33
Malicious artifacts: 51


In [33]:
# Identify endpoints that exist in the graph but have no explicit malicious
# evidence on any of their observed artifact relationships.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)

    OPTIONAL MATCH (e)-[r:OBSERVED]->(a:Artifact)

    WITH
        e,
        collect(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
                ELSE 0
            END
        ) AS malicious_flags

    RETURN
        count(
            CASE
                WHEN all(flag IN malicious_flags WHERE flag = 0)
                THEN e
            END
        ) AS endpoints_without_malicious_evidence
    """,
    database_=DATABASE,
)

print(
    "Endpoints without explicit malicious evidence:",
    result.records[0]["endpoints_without_malicious_evidence"],
)

Endpoints without explicit malicious evidence: 214


In [34]:
# Retrieve all final graph metrics in one query so the expected project
# population can be checked in a single validation step.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    WITH count(e) AS endpoints

    MATCH (a:Artifact)
    WITH
        endpoints,
        count(a) AS artifacts

    MATCH ()-[r:OBSERVED]->()
    WITH
        endpoints,
        artifacts,
        count(r) AS relationships

    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WITH
        endpoints,
        artifacts,
        relationships,
        count(
            CASE
                WHEN r.malicious_detected = true
                THEN 1
            END
        ) AS malicious_relationships,
        count(
            DISTINCT CASE
                WHEN r.malicious_detected = true
                THEN e
            END
        ) AS malicious_endpoints

    RETURN
        endpoints,
        artifacts,
        relationships,
        malicious_relationships,
        malicious_endpoints
    """,
    database_=DATABASE,
)

final_metrics = dict(result.records[0])


print("=" * 70)
print("FINAL GRAPH VALIDATION")
print("=" * 70)

for key, value in final_metrics.items():
    print(f"{key}: {value}")


# These assertions protect the project from silently importing the wrong
# dataset or leaving stale graph data behind.
assert final_metrics["endpoints"] == 247, (
    "The graph does not contain the expected 247 endpoints."
)

assert final_metrics["relationships"] == 72, (
    "The graph does not contain the expected 72 OBSERVED relationships."
)

assert final_metrics["malicious_relationships"] == 60, (
    "The graph does not contain the expected 60 malicious relationships."
)

assert final_metrics["malicious_endpoints"] == 33, (
    "The graph does not contain the expected 33 malicious endpoints."
)

print("\nAll final graph validation checks passed.")

FINAL GRAPH VALIDATION
endpoints: 247
artifacts: 60
relationships: 72
malicious_relationships: 60
malicious_endpoints: 33

All final graph validation checks passed.


In [35]:
# Inspect one known endpoint with explicit malicious evidence.
# This provides a concrete forensic validation example for the project.
result = driver.execute_query(
    """
    MATCH (e:Endpoint {
        endpoint_id: 'EPVSWBNA'
    })
    OPTIONAL MATCH (e)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count,
        r.malicious_detected AS malicious_detected,
        r.malicious_detection_count AS detection_count,
        r.malicious_file_names AS malicious_file_names,
        r.first_malicious_detection AS first_detection,
        r.last_malicious_detection AS last_detection

    ORDER BY r.first_malicious_detection
    """,
    database_=DATABASE,
)

epvswbna_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

display(epvswbna_df)

,endpoint,sha256,file_names,record_count,malicious_detected,detection_count,malicious_file_names,first_detection,last_detection
0,EPVSWBNA,61a67e53c6c0c80a4e95af35a2997ad2ce59720020fc74...,[launcher.exe],1,True,1,[launcher.exe],2026-09-24T12:10:16.000000000,2026-09-24T12:10:16.000000000
1,EPVSWBNA,ae107170c153cb3eb34c049482722d3937f2c03a6c23ca...,[Strings.dll],1,True,1,[Strings.dll],2026-09-24T12:10:16.000000000,2026-09-24T12:10:16.000000000
2,EPVSWBNA,c6525c5230387653e16f21f408eab8202157b0eddeb180...,[zshp1020s.dll],1,True,1,[zshp1020s.dll],2026-09-24T12:10:16.000000000,2026-09-24T12:10:16.000000000


In [36]:
# Find an endpoint that exists in the raw population but has no explicit
# malicious evidence, demonstrating that the graph does not classify
# every endpoint as malicious.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)

    WHERE NOT EXISTS {
        MATCH (e)-[r:OBSERVED]->()
        WHERE r.malicious_detected = true
    }

    RETURN e.endpoint_id AS endpoint
    ORDER BY endpoint
    LIMIT 1
    """,
    database_=DATABASE,
)

if result.records:
    example_endpoint = result.records[0]["endpoint"]

    print(
        "Example endpoint without explicit malicious evidence:",
        example_endpoint,
    )
else:
    print(
        "No endpoint without explicit malicious evidence was found."
    )

Example endpoint without explicit malicious evidence: EP2C4ZH5


In [37]:
# Close the Neo4j driver cleanly after all import and validation work.
driver.close()

print("Neo4j driver closed.")

Neo4j driver closed.


In [38]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        r.record_count AS record_count,
        r.first_event AS first_event,
        r.last_event AS last_event
    ORDER BY endpoint, sha256
    LIMIT 10
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='100d970fc784e9bcb37b707ec4ef69825b165a073659f4068070a165184a087f' record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='39f5d83d7982cfae057053b0dcffa66fca12bee9769b1587a27e4387f56f29ec' record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785e4a28dfb7fc8032fc4' record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553756bf8ad8c7dfaa2bd' record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='528c866fabbe17c30e9790a12ac09b94dd820ba

In [39]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    WITH count(e) AS endpoints

    MATCH (a:Artifact)
    WITH endpoints, count(a) AS artifacts

    MATCH ()-[r:OBSERVED]->()
    RETURN
        endpoints,
        artifacts,
        count(r) AS observed_relationships
    """,
    database_=DATABASE
)

print(result.records[0])

<Record endpoints=22 artifacts=49 observed_relationships=49>


In [40]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    RETURN
        count(*) AS relationships,
        count(DISTINCT e) AS connected_endpoints,
        count(DISTINCT a) AS connected_artifacts
    """,
    database_=DATABASE
)

print(result.records[0])


<Record relationships=49 connected_endpoints=22 connected_artifacts=49>


In [41]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[:OBSERVED]->(a:Artifact)

    WITH a, count(e) AS endpoint_count

    WHERE endpoint_count > 1

    RETURN
        a.sha256 AS sha256,
        endpoint_count,
        a.file_names AS file_names

    ORDER BY endpoint_count DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

In [42]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count,
        r.first_event AS first_event,
        r.last_event AS last_event

    ORDER BY endpoint, r.first_event
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' file_names=['load-hub-i18n.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='100d970fc784e9bcb37b707ec4ef69825b165a073659f4068070a165184a087f' file_names=['shopping_iframe_driver.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='39f5d83d7982cfae057053b0dcffa66fca12bee9769b1587a27e4387f56f29ec' file_names=['tokenized-card.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785e4a28dfb7fc8032fc4' file_names=['wallet_checkout_autofill_driver.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553756bf8ad

In [43]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        count(DISTINCT a) AS artifact_count,
        sum(r.record_count) AS total_records

    ORDER BY artifact_count DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' artifact_count=20 total_records=40>
<Record endpoint='EPMVZYZO' artifact_count=4 total_records=4>
<Record endpoint='EPVSWBNA' artifact_count=3 total_records=3>
<Record endpoint='EPAJI3E5' artifact_count=2 total_records=4>
<Record endpoint='EPJ42VCZ' artifact_count=2 total_records=2>
<Record endpoint='EPPGWIOF' artifact_count=2 total_records=10>
<Record endpoint='EP7UYCXM' artifact_count=1 total_records=1>
<Record endpoint='EPAGTKNF' artifact_count=1 total_records=2>
<Record endpoint='EPEEHWSG' artifact_count=1 total_records=1>
<Record endpoint='EPGSQVEB' artifact_count=1 total_records=1>
<Record endpoint='EPJ76RZV' artifact_count=1 total_records=4>
<Record endpoint='EPMDU6L5' artifact_count=1 total_records=1>
<Record endpoint='EPNPX6VY' artifact_count=1 total_records=1>
<Record endpoint='EPQBJBDX' artifact_count=1 total_records=1>
<Record endpoint='EPQKOBYG' artifact_count=1 total_records=1>
<Record endpoint='EPRD7FJT' artifact_count=1 total_records=1>
<Reco

In [44]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count

    ORDER BY r.record_count DESC
    LIMIT 20
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EPPGWIOF' sha256='6594cac08d03782ad43d6b293faff904f20032c8eb37a1b27b6caf57b1aeae5e' file_names=['brprintconflsr3'] record_count=5>
<Record endpoint='EPPGWIOF' sha256='64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d92358b64771980665af' file_names=['rawtobr3'] record_count=5>
<Record endpoint='EPSES7ZK' sha256='f0c485571570100898f2d15531fbe4531ba303a252cd40a79b64c1976a5a9890' file_names=['hp_LJ1020_Full_Solution-v2012_918_1_57980(1).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(2).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(3).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(4).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980.exe'] record_count=5>
<Record endpoint='EPJ76RZV' sha256='d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f811c5d94fa6ce8b7ef56' file_names=['COMPRESSED FINAL COMPARISION RETIREES WELFARE1.xlsx'] record_count=4>
<Record endpoint='EP4YIFHV' sha256='528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f9d9bb78a2795992f1c' file_names=['wallet_donation_driver

In [45]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        r.first_event AS first_event,
        r.last_event AS last_event,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count

    ORDER BY endpoint, first_event, last_event
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09' sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' file_names=['load-hub-i18n.bundle.js'] record_count=2>
<Record endpoint='EP4YIFHV' first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09' sha256='100d970fc784e9bcb37b707ec4ef69825b165a073659f4068070a165184a087f' file_names=['shopping_iframe_driver.js'] record_count=2>
<Record endpoint='EP4YIFHV' first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09' sha256='39f5d83d7982cfae057053b0dcffa66fca12bee9769b1587a27e4387f56f29ec' file_names=['tokenized-card.bundle.js'] record_count=2>
<Record endpoint='EP4YIFHV' first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09' sha256='444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785e4a28dfb7fc8032fc4' file_names=['wallet_checkout_autofill_driver.js'] record_count=2>
<Record endpoint='EP4YIFHV' first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35

In [46]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.first_event AS first_event,
        r.last_event AS last_event,
        r.record_count AS record_count

    ORDER BY r.record_count DESC, r.first_event
    LIMIT 20
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EPPGWIOF' sha256='64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d92358b64771980665af' file_names=['rawtobr3'] first_event='2026-09-07 11:25:38' last_event='2026-09-08 08:12:56' record_count=5>
<Record endpoint='EPPGWIOF' sha256='6594cac08d03782ad43d6b293faff904f20032c8eb37a1b27b6caf57b1aeae5e' file_names=['brprintconflsr3'] first_event='2026-09-07 11:25:38' last_event='2026-09-08 08:12:56' record_count=5>
<Record endpoint='EPSES7ZK' sha256='f0c485571570100898f2d15531fbe4531ba303a252cd40a79b64c1976a5a9890' file_names=['hp_LJ1020_Full_Solution-v2012_918_1_57980(1).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(2).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(3).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(4).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980.exe'] first_event='2026-09-22 08:01:03' last_event='2026-09-22 08:01:03' record_count=5>
<Record endpoint='EPJ76RZV' sha256='d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f811c5d94fa6ce8b7ef56' file_names=['COMPRE

In [47]:
endpoint_id = "EP4YIFHV"

result = driver.execute_query(
    """
    MATCH (e:Endpoint {endpoint_id: $endpoint_id})
          -[r:OBSERVED]->
          (a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count,
        r.first_event AS first_event,
        r.last_event AS last_event

    ORDER BY r.first_event
    """,
    endpoint_id=endpoint_id,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' file_names=['load-hub-i18n.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='100d970fc784e9bcb37b707ec4ef69825b165a073659f4068070a165184a087f' file_names=['shopping_iframe_driver.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='39f5d83d7982cfae057053b0dcffa66fca12bee9769b1587a27e4387f56f29ec' file_names=['tokenized-card.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785e4a28dfb7fc8032fc4' file_names=['wallet_checkout_autofill_driver.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553756bf8ad

In [48]:
sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10'

result = driver.execute_query(
    """
    MATCH (a:Artifact)
    RETURN a.sha256 AS sha256
    LIMIT 1
    """,
    database_=DATABASE
)

sha256 = result.records[0]["sha256"]

print("Selected artifact:", sha256)

Selected artifact: 036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10


In [49]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact {sha256: $sha256})

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count,
        r.first_event AS first_event,
        r.last_event AS last_event

    ORDER BY r.first_event
    """,
    sha256=sha256,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' file_names=['load-hub-i18n.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>


In [50]:
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    WHERE a.file_name_count > 1

    RETURN
        a.sha256 AS sha256,
        a.file_names AS file_names,
        a.file_name_count AS file_name_count,
        a.endpoint_count AS endpoint_count,
        a.record_count AS record_count

    ORDER BY a.file_name_count DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record sha256='f0c485571570100898f2d15531fbe4531ba303a252cd40a79b64c1976a5a9890' file_names=['hp_LJ1020_Full_Solution-v2012_918_1_57980(1).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(2).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(3).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(4).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980.exe'] file_name_count=5 endpoint_count=1 record_count=5>


In [51]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        a.sha256 AS sha256,
        a.file_names AS file_names,
        sum(r.record_count) AS total_observations,
        count(DISTINCT e) AS endpoint_count

    ORDER BY total_observations DESC
    LIMIT 20
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record sha256='6594cac08d03782ad43d6b293faff904f20032c8eb37a1b27b6caf57b1aeae5e' file_names=['brprintconflsr3'] total_observations=5 endpoint_count=1>
<Record sha256='64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d92358b64771980665af' file_names=['rawtobr3'] total_observations=5 endpoint_count=1>
<Record sha256='f0c485571570100898f2d15531fbe4531ba303a252cd40a79b64c1976a5a9890' file_names=['hp_LJ1020_Full_Solution-v2012_918_1_57980(1).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(2).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(3).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980(4).exe', 'hp_LJ1020_Full_Solution-v2012_918_1_57980.exe'] total_observations=5 endpoint_count=1>
<Record sha256='d4a23078663777e758c04d2670b0d6aac8aa7ba5f91f811c5d94fa6ce8b7ef56' file_names=['COMPRESSED FINAL COMPARISION RETIREES WELFARE1.xlsx'] total_observations=4 endpoint_count=1>
<Record sha256='528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f9d9bb78a2795992f1c' file_names=['wallet_donation_driver.js'] to

In [52]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        count(DISTINCT a) AS unique_artifacts,
        sum(r.record_count) AS total_observations,
        min(r.first_event) AS first_observation,
        max(r.last_event) AS last_observation

    ORDER BY total_observations DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' unique_artifacts=20 total_observations=40 first_observation='2026-09-11 09:23:33' last_observation='2026-09-11 09:35:09'>
<Record endpoint='EPPGWIOF' unique_artifacts=2 total_observations=10 first_observation='2026-09-07 11:25:38' last_observation='2026-09-08 08:12:56'>
<Record endpoint='EPSES7ZK' unique_artifacts=1 total_observations=5 first_observation='2026-09-22 08:01:03' last_observation='2026-09-22 08:01:03'>
<Record endpoint='EPAJI3E5' unique_artifacts=2 total_observations=4 first_observation='2026-09-17 16:43:44' last_observation='2026-09-18 08:28:01'>
<Record endpoint='EPJ76RZV' unique_artifacts=1 total_observations=4 first_observation='2026-09-14 13:30:13' last_observation='2026-09-21 15:12:40'>
<Record endpoint='EPMVZYZO' unique_artifacts=4 total_observations=4 first_observation='2026-09-13 17:27:55' last_observation='2026-09-13 17:27:55'>
<Record endpoint='EPVSWBNA' unique_artifacts=3 total_observations=3 first_observation='2026-09-24 12:10:16' l

In [53]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    WITH
        e,
        min(r.first_event) AS first_event,
        max(r.last_event) AS last_event,
        count(DISTINCT a) AS artifact_count,
        sum(r.record_count) AS total_observations

    RETURN
        e.endpoint_id AS endpoint,
        artifact_count,
        total_observations,
        first_event,
        last_event

    ORDER BY artifact_count DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

<Record endpoint='EP4YIFHV' artifact_count=20 total_observations=40 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EPMVZYZO' artifact_count=4 total_observations=4 first_event='2026-09-13 17:27:55' last_event='2026-09-13 17:27:55'>
<Record endpoint='EPVSWBNA' artifact_count=3 total_observations=3 first_event='2026-09-24 12:10:16' last_event='2026-09-24 12:10:16'>
<Record endpoint='EPAJI3E5' artifact_count=2 total_observations=4 first_event='2026-09-17 16:43:44' last_event='2026-09-18 08:28:01'>
<Record endpoint='EPJ42VCZ' artifact_count=2 total_observations=2 first_event='2026-09-07 08:43:44' last_event='2026-09-07 08:43:44'>
<Record endpoint='EPPGWIOF' artifact_count=2 total_observations=10 first_event='2026-09-07 11:25:38' last_event='2026-09-08 08:12:56'>
<Record endpoint='EP7UYCXM' artifact_count=1 total_observations=1 first_event='2026-09-26 15:25:25' last_event='2026-09-26 15:25:25'>
<Record endpoint='EPAGTKNF' artifact_count=1 total_observat

In [54]:
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    UNWIND a.file_names AS file_name

    WITH file_name, collect(a.sha256) AS artifacts

    WHERE size(artifacts) > 1

    RETURN
        file_name,
        artifacts,
        size(artifacts) AS artifact_count

    ORDER BY artifact_count DESC, file_name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

In [55]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[:OBSERVED]->(a:Artifact)
    UNWIND a.file_names AS file_name

    WITH
        file_name,
        collect(DISTINCT a.sha256) AS artifacts,
        collect(DISTINCT e.endpoint_id) AS endpoints

    WHERE size(artifacts) > 1

    RETURN
        file_name,
        artifacts,
        endpoints,
        size(artifacts) AS artifact_count,
        size(endpoints) AS endpoint_count

    ORDER BY artifact_count DESC, file_name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record)

In [56]:
def investigate_endpoint(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
              -[r:OBSERVED]->
              (a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            a.sha256 AS sha256,
            a.file_names AS file_names,
            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event

        ORDER BY r.first_event, a.sha256
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    return result.records

In [57]:
records = investigate_endpoint("EP4YIFHV")

for record in records:
    print(record)

<Record endpoint='EP4YIFHV' sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' file_names=['load-hub-i18n.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='100d970fc784e9bcb37b707ec4ef69825b165a073659f4068070a165184a087f' file_names=['shopping_iframe_driver.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='39f5d83d7982cfae057053b0dcffa66fca12bee9769b1587a27e4387f56f29ec' file_names=['tokenized-card.bundle.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785e4a28dfb7fc8032fc4' file_names=['wallet_checkout_autofill_driver.js'] record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>
<Record endpoint='EP4YIFHV' sha256='4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553756bf8ad

In [58]:
investigate_endpoint("EPAGTKNF")

[<Record endpoint='EPAGTKNF' sha256='18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3d2d6331b90e9162208' file_names=['Buy CP PLUS Dashboard Camera, Warranty NONE 1 year online _ Public Procurement Portal (PPP).pdf'] record_count=2 first_event='2026-09-03 19:03:33' last_event='2026-09-04 18:10:01'>]

In [59]:
def investigate_artifact(sha256):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint)-[r:OBSERVED]->
              (a:Artifact {sha256: $sha256})

        RETURN
            a.sha256 AS sha256,
            a.file_names AS file_names,
            e.endpoint_id AS endpoint,
            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event

        ORDER BY r.first_event, e.endpoint_id
        """,
        sha256=sha256,
        database_=DATABASE
    )

    return result.records

In [60]:
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    RETURN a.sha256 AS sha256
    LIMIT 1
    """,
    database_=DATABASE
)

test_sha256 = result.records[0]["sha256"]

print(test_sha256)

036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10


In [61]:
records = investigate_artifact(test_sha256)

for record in records:
    print(record)

<Record sha256='036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10' file_names=['load-hub-i18n.bundle.js'] endpoint='EP4YIFHV' record_count=2 first_event='2026-09-11 09:23:33' last_event='2026-09-11 09:35:09'>


In [62]:
def endpoint_report(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
              -[r:OBSERVED]->
              (a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            count(DISTINCT a) AS unique_artifacts,
            sum(r.record_count) AS total_observations,
            min(r.first_event) AS first_observation,
            max(r.last_event) AS last_observation
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    if not result.records:
        return None

    return dict(result.records[0])

In [63]:
report = endpoint_report("EP4YIFHV")
print(report)

{'endpoint': 'EP4YIFHV', 'unique_artifacts': 20, 'total_observations': 40, 'first_observation': '2026-09-11 09:23:33', 'last_observation': '2026-09-11 09:35:09'}


In [64]:
def artifact_report(sha256):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint)-[r:OBSERVED]->
              (a:Artifact {sha256: $sha256})

        RETURN
            a.sha256 AS sha256,
            a.file_names AS file_names,
            a.record_count AS artifact_record_count,
            a.file_name_count AS file_name_count,
            count(DISTINCT e) AS endpoint_count,
            collect({
                endpoint: e.endpoint_id,
                record_count: r.record_count,
                first_event: r.first_event,
                last_event: r.last_event
            }) AS observations
        """,
        sha256=sha256,
        database_=DATABASE
    )

    if not result.records:
        return None

    return dict(result.records[0])

In [65]:
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    RETURN a.sha256 AS sha256
    LIMIT 1
    """,
    database_=DATABASE
)

test_sha256 = result.records[0]["sha256"]

print(artifact_report(test_sha256))

{'sha256': '036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10', 'file_names': ['load-hub-i18n.bundle.js'], 'artifact_record_count': 2, 'file_name_count': 1, 'endpoint_count': 1, 'observations': [{'record_count': 2, 'endpoint': 'EP4YIFHV', 'last_event': '2026-09-11 09:35:09', 'first_event': '2026-09-11 09:23:33'}]}


In [66]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.record_count AS record_count,
        r.first_event AS first_event,
        r.last_event AS last_event

    ORDER BY endpoint, first_event
    """,
    database_=DATABASE
)

investigation_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

display(investigation_df)

,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [67]:
print("Investigation rows:", len(investigation_df))
print("Endpoints:", investigation_df["endpoint"].nunique())
print("Artifacts:", investigation_df["sha256"].nunique())

Investigation rows: 49
Endpoints: 22
Artifacts: 49


In [68]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)

    RETURN
        e.endpoint_id AS endpoint,
        count(DISTINCT a) AS unique_artifacts,
        sum(r.record_count) AS total_observations,
        min(r.first_event) AS first_observation,
        max(r.last_event) AS last_observation

    ORDER BY endpoint
    """,
    database_=DATABASE
)

endpoint_summary_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

display(endpoint_summary_df)

,endpoint,unique_artifacts,total_observations,first_observation,last_observation
0,EP4YIFHV,20,40,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP7UYCXM,1,1,2026-09-26 15:25:25,2026-09-26 15:25:25
2,EPAGTKNF,1,2,2026-09-03 19:03:33,2026-09-04 18:10:01
3,EPAJI3E5,2,4,2026-09-17 16:43:44,2026-09-18 08:28:01
4,EPEEHWSG,1,1,2026-09-10 09:15:06,2026-09-10 09:15:06
5,EPGSQVEB,1,1,2026-09-11 10:11:45,2026-09-11 10:11:45
6,EPJ42VCZ,2,2,2026-09-07 08:43:44,2026-09-07 08:43:44
7,EPJ76RZV,1,4,2026-09-14 13:30:13,2026-09-21 15:12:40
8,EPMDU6L5,1,1,2026-09-08 11:58:09,2026-09-08 11:58:09
9,EPMVZYZO,4,4,2026-09-13 17:27:55,2026-09-13 17:27:55


In [69]:
def investigate_endpoint(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
              -[r:OBSERVED]->
              (a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            a.sha256 AS sha256,
            a.file_names AS file_names,
            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event

        ORDER BY r.first_event, r.last_event
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [dict(record) for record in result.records]
    )

In [70]:
df = investigate_endpoint("EP4YIFHV")

print("Rows:", len(df))
display(df)

Rows: 20


,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [71]:
def endpoint_exists(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
        RETURN count(e) AS count
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    return result.records[0]["count"] == 1

In [72]:
print(endpoint_exists("EP4YIFHV"))

True


In [73]:
print(endpoint_exists("INVALID_ENDPOINT"))

False


In [74]:
def investigate(endpoint_id):
    if not endpoint_exists(endpoint_id):
        raise ValueError(
            f"Endpoint '{endpoint_id}' does not exist in the graph."
        )

    df = investigate_endpoint(endpoint_id)

    summary = {
        "endpoint": endpoint_id,
        "unique_artifacts": df["sha256"].nunique(),
        "total_records": int(df["record_count"].sum()),
        "first_observation": df["first_event"].min(),
        "last_observation": df["last_event"].max(),
    }

    return summary, df

In [75]:
summary, artifacts = investigate("EP4YIFHV")

print(summary)
display(artifacts)

{'endpoint': 'EP4YIFHV', 'unique_artifacts': 20, 'total_records': 40, 'first_observation': '2026-09-11 09:23:33', 'last_observation': '2026-09-11 09:35:09'}


,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [76]:
def investigate_artifact(sha256):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint)-[r:OBSERVED]->
              (a:Artifact {sha256: $sha256})

        RETURN
            e.endpoint_id AS endpoint,
            a.sha256 AS sha256,
            a.file_names AS file_names,
            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event

        ORDER BY r.first_event
        """,
        sha256=sha256,
        database_=DATABASE
    )

    return pd.DataFrame(
        [dict(record) for record in result.records]
    )

In [77]:
result = driver.execute_query(
    """
    MATCH (a:Artifact)
    RETURN a.sha256 AS sha256
    LIMIT 1
    """,
    database_=DATABASE
)

sha256 = result.records[0]["sha256"]

print("Testing artifact:", sha256)

Testing artifact: 036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10


In [78]:
artifact_df = investigate_artifact(sha256)

display(artifact_df)

,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [79]:
result = driver.execute_query(
    """
    MATCH (e:Endpoint)
    WITH count(e) AS endpoints

    MATCH (a:Artifact)
    WITH endpoints, count(a) AS artifacts

    MATCH ()-[r:OBSERVED]->()
    RETURN
        endpoints,
        artifacts,
        count(r) AS relationships
    """,
    database_=DATABASE
)

record = result.records[0]

print("Endpoints:", record["endpoints"])
print("Artifacts:", record["artifacts"])
print("OBSERVED relationships:", record["relationships"])

Endpoints: 22
Artifacts: 49
OBSERVED relationships: 49


In [80]:
def investigate_endpoint(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
              -[r:OBSERVED]->
              (a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            a.sha256 AS sha256,
            a.file_names AS file_names,
            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event

        ORDER BY r.first_event, r.last_event
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    if not result.records:
        return None

    return pd.DataFrame(
        [dict(record) for record in result.records]
    )

In [81]:
df = investigate_endpoint("EP4YIFHV")

display(df)

,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [82]:
def endpoint_summary(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
              -[r:OBSERVED]->
              (a:Artifact)

        RETURN
            e.endpoint_id AS endpoint,
            count(DISTINCT a) AS unique_artifacts,
            sum(r.record_count) AS total_observations,
            min(r.first_event) AS first_observation,
            max(r.last_event) AS last_observation
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    if not result.records:
        return None

    return dict(result.records[0])

In [83]:
summary = endpoint_summary("EP4YIFHV")

summary

{'endpoint': 'EP4YIFHV',
 'unique_artifacts': 20,
 'total_observations': 40,
 'first_observation': '2026-09-11 09:23:33',
 'last_observation': '2026-09-11 09:35:09'}

In [84]:
def investigate(endpoint_id):
    summary = endpoint_summary(endpoint_id)
    artifacts = investigate_endpoint(endpoint_id)

    if summary is None:
        return None

    return {
        "summary": summary,
        "artifacts": artifacts
    }

In [85]:
report = investigate("EP4YIFHV")

print(report["summary"])
display(report["artifacts"])

{'endpoint': 'EP4YIFHV', 'unique_artifacts': 20, 'total_observations': 40, 'first_observation': '2026-09-11 09:23:33', 'last_observation': '2026-09-11 09:35:09'}


,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [86]:
def endpoint_exists(endpoint_id):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint {endpoint_id: $endpoint_id})
        RETURN count(e) AS count
        """,
        endpoint_id=endpoint_id,
        database_=DATABASE
    )

    return result.records[0]["count"] == 1

In [87]:
def investigate(endpoint_id):
    if not endpoint_exists(endpoint_id):
        print(f"Endpoint not found: {endpoint_id}")
        return None

    summary = endpoint_summary(endpoint_id)
    artifacts = investigate_endpoint(endpoint_id)

    return {
        "summary": summary,
        "artifacts": artifacts
    }

In [88]:
report = investigate("EP4YIFHV")

print("Endpoint Summary")
print("----------------")
for key, value in report["summary"].items():
    print(f"{key}: {value}")

print("\nArtifacts")
print("---------")
display(report["artifacts"])

Endpoint Summary
----------------
endpoint: EP4YIFHV
unique_artifacts: 20
total_observations: 40
first_observation: 2026-09-11 09:23:33
last_observation: 2026-09-11 09:35:09

Artifacts
---------


,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,[runtime.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,[load-ec-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,[bnpl.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,[bnpl_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [89]:
report = investigate("INVALID_ENDPOINT")

print(report)

Endpoint not found: INVALID_ENDPOINT
None


In [90]:
def investigate_artifact(sha256):
    result = driver.execute_query(
        """
        MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact {sha256: $sha256})

        RETURN
            e.endpoint_id AS endpoint,
            a.sha256 AS sha256,
            a.file_names AS file_names,
            r.record_count AS record_count,
            r.first_event AS first_event,
            r.last_event AS last_event

        ORDER BY r.first_event
        """,
        sha256=sha256,
        database_=DATABASE
    )

    if not result.records:
        return None

    return pd.DataFrame(
        [dict(record) for record in result.records]
    )

In [91]:
sha256 = "036965c24ac54284e0bfac480395c91214e7b3c64d00215266a75e8977140f10"

artifact_df = investigate_artifact(sha256)

display(artifact_df)

,endpoint,sha256,file_names,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09


In [92]:
def artifact_exists(sha256):
    result = driver.execute_query(
        """
        MATCH (a:Artifact {sha256: $sha256})
        RETURN count(a) AS count
        """,
        sha256=sha256,
        database_=DATABASE
    )

    return result.records[0]["count"] == 1

In [93]:
def investigate_artifact_safe(sha256):
    if not artifact_exists(sha256):
        print(f"Artifact not found: {sha256}")
        return None

    return investigate_artifact(sha256)

In [94]:
result = investigate_artifact_safe("INVALID_SHA256")

print(result)

Artifact not found: INVALID_SHA256
None



## Malicious Evidence Graph Validation

The following query verifies that malicious detection metadata was stored on
the endpoint-specific `OBSERVED` relationships.


In [ ]:

# Verify that the graph contains endpoint-artifact relationships carrying
# explicit malicious security evidence.
result = driver.execute_query(
    """
    MATCH (e:Endpoint)-[r:OBSERVED]->(a:Artifact)
    WHERE r.malicious_detected = true

    RETURN
        e.endpoint_id AS endpoint,
        a.sha256 AS sha256,
        a.file_names AS file_names,
        r.malicious_detection_count AS detection_count,
        r.malicious_file_names AS malicious_file_names,
        r.first_malicious_detection AS first_detection,
        r.last_malicious_detection AS last_detection,
        r.first_quarantine_time AS first_quarantine,
        r.last_quarantine_time AS last_quarantine

    ORDER BY r.first_malicious_detection
    """,
    database_=DATABASE
)

malicious_graph_df = pd.DataFrame(
    [dict(record) for record in result.records]
)

print(
    "Graph relationships with explicit malicious evidence:",
    len(malicious_graph_df)
)

display(malicious_graph_df.head(20))


Graph relationships with explicit malicious evidence: 49


,endpoint,sha256,file_names,detection_count,malicious_file_names,first_detection,last_detection,first_quarantine,last_quarantine
0,EPZRAPEL,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,[53EDA42869D426262B752F088EA605187284ACE8],1,[['53EDA42869D426262B752F088EA605187284ACE8']],2026-09-01 04:58:58,2026-09-01 04:58:58,2026-08-31 13:21:00,2026-08-31 13:21:00
1,EPAGTKNF,18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"[Buy CP PLUS Dashboard Camera, Warranty NONE 1...",2,"[['Buy CP PLUS Dashboard Camera, Warranty NONE...",2026-09-03 19:03:33,2026-09-04 18:10:01,2026-09-03 11:25:00,2026-09-03 11:25:00
2,EPJ42VCZ,854b8f7716627df333e8ecfeae5887c02b3195658f7394...,[RAGHAV STNA STNB.pdf],1,[['RAGHAV STNA STNB.pdf']],2026-09-07 08:43:44,2026-09-07 08:43:44,2026-09-06 10:32:00,2026-09-06 10:32:00
3,EPJ42VCZ,a2beca1870ae689f4b046a20c3d8e2e48557d782dea534...,[RAGHAV STNC STNA.pdf],1,[['RAGHAV STNC STNA.pdf']],2026-09-07 08:43:44,2026-09-07 08:43:44,2026-09-06 10:20:00,2026-09-06 10:20:00
4,EPPGWIOF,64efda7704564dd50c7514e74ef3c5252bc68af8f3b0d9...,[rawtobr3],5,[['rawtobr3']],2026-09-07 11:25:38,2026-09-08 08:12:56,2026-09-07 10:58:00,2026-09-07 10:58:00
5,EPPGWIOF,6594cac08d03782ad43d6b293faff904f20032c8eb37a1...,[brprintconflsr3],5,[['brprintconflsr3']],2026-09-07 11:25:38,2026-09-08 08:12:56,2026-09-07 10:57:00,2026-09-07 10:57:00
6,EPQKOBYG,f475a7462dd85932ad7f99354751663d1543adb5e37c7c...,[win-lbp623-621-fw-v1301(1).exe],1,[['win-lbp623-621-fw-v1301(1).exe']],2026-09-08 08:31:09,2026-09-08 08:31:09,2026-09-07 13:57:00,2026-09-07 13:57:00
7,EPMDU6L5,5dd59e8309b4be90aad58871d82a5b3fb6dcd7ff19a7fb...,[recently-used.xbel],1,[['recently-used.xbel']],2026-09-08 11:58:09,2026-09-08 11:58:09,2026-05-13 13:24:00,2026-05-13 13:24:00
8,EPEEHWSG,98d119993c27068f00bdf18a17d6613ae6ce6046b6b662...,[Library Books.xlsx],1,[['Library Books.xlsx']],2026-09-10 09:15:06,2026-09-10 09:15:06,2026-09-08 21:19:00,2026-09-08 21:19:00
9,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,[['load-hub-i18n.bundle.js']],2026-09-11 09:23:33,2026-09-11 09:35:09,2026-09-10 13:30:00,2026-09-10 13:30:00
